# 02. Extract PCAOB Deficiency Data and SEC EDGAR Filings

Pulls real audit-deficiency data from PCAOB's official downloadable inspection
datasets, and real material-weakness disclosure data from SEC EDGAR's Full-Text
Search system.

**Output:**
- `data/raw/pcaob_deficiencies_raw.csv`
- `data/raw/sec_edgar_disclosures_raw.json`

**Status — two very different states for the two halves of this script:**
- **PCAOB half: done, for real.** The real file (`pcaob_deficiencies_raw.csv`,
  16,704 real deficiency records) already exists in this repo. PCAOB does not
  offer a simple stable API — their bulk datasets are downloaded manually from
  their Firm Inspection Reports page, which is what happened here.
- **SEC EDGAR half: genuinely not yet run.** No `sec_edgar_disclosures_raw.json`
  exists in this repo yet. This part is still needed for RQ4 (remediation-time
  modeling) and remains an open task.


In [1]:
!pip install -q requests pandas || pip install -q requests pandas --break-system-packages

## Part 1: PCAOB inspection datasets (already completed)

In [2]:
import pandas as pd
import os, urllib.request

GITHUB_BASE = "https://raw.githubusercontent.com/daljeetkaurJohar/qm640-governance-analytics/master"

def ensure_file(local_path, github_relative_path):
    if os.path.exists(local_path):
        print(f"Found locally: {local_path}")
        return True
    os.makedirs(os.path.dirname(local_path) or ".", exist_ok=True)
    url = f"{GITHUB_BASE}/{github_relative_path}"
    try:
        print(f"Not found locally -- fetching from GitHub repo: {url}")
        req = urllib.request.Request(url, headers={"User-Agent": "qm640-capstone"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            content = resp.read()
        with open(local_path, "wb") as f:
            f.write(content)
        print(f"Downloaded {len(content)} bytes from GitHub -> {local_path}")
        return True
    except Exception as e:
        print(f"GitHub fetch failed ({e}).")
        return False

# PCAOB publishes bulk CSV/XML/JSON files directly -- no API call needed.
# Real download page: https://pcaobus.org/oversight/inspections/firm-inspection-reports
# The real file is already uploaded to this project's GitHub repo, so it is
# fetched from there directly if not already present locally.

PCAOB_PATH = "../data/raw/pcaob_deficiencies_raw.csv"
ensure_file(PCAOB_PATH, "data/raw/pcaob_deficiencies_raw.csv")

if os.path.exists(PCAOB_PATH):
    pcaob_df = pd.read_csv(PCAOB_PATH)
    print(f"Real PCAOB data present: {len(pcaob_df)} deficiency records")
    print(pcaob_df["severity_part"].value_counts())
else:
    print("PCAOB file not found locally or on GitHub -- download manually from:")
    print("https://pcaobus.org/oversight/inspections/firm-inspection-reports")
    print("then save it to", PCAOB_PATH)

Not found locally -- fetching from GitHub repo: https://raw.githubusercontent.com/daljeetkaurJohar/qm640-governance-analytics/master/data/raw/pcaob_deficiencies_raw.csv
Downloaded 11214050 bytes from GitHub -> ../data/raw/pcaob_deficiencies_raw.csv
Real PCAOB data present: 16704 deficiency records
severity_part
I.A    13743
I.B     2961
Name: count, dtype: int64


## Part 2: SEC EDGAR material weakness disclosures (2020-2026) — NOT YET RUN

This part is genuinely untested. SEC's full-text search endpoint requires a
descriptive `User-Agent` header per their fair-access policy (mandatory, not
optional). Running this against the live API is the immediate next step needed
for RQ4.

In [3]:
import requests
import json
import time

EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"

HEADERS = {
    "User-Agent": "Daljeet Kaur Johar daljeetkaur07@gmail.com"
}


def search_material_weakness_filings(form_type: str, start_date: str, end_date: str,
                                      max_pages: int = 10) -> list:
    """
    Search SEC EDGAR full-text search for material weakness disclosures.
    form_type: e.g. "8-K" or "10-K"
    start_date / end_date: "YYYY-MM-DD"
    """
    all_hits = []
    for page in range(max_pages):
        params = {
            "q": '"material weakness"',
            "forms": form_type,
            "dateRange": "custom",
            "startdt": start_date,
            "enddt": end_date,
            "from": page * 10,
        }
        response = requests.get(EDGAR_SEARCH_URL, params=params, headers=HEADERS, timeout=30)
        if response.status_code != 200:
            print(f"  Stopped at page {page}, status {response.status_code}")
            break

        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        if not hits:
            break

        all_hits.extend(hits)
        print(f"  {form_type}: fetched {len(all_hits)} filings so far...")
        time.sleep(0.3)  # respect SEC's rate limits

    return all_hits

In [4]:
# NOT EXECUTED HERE -- this notebook's sandbox cannot reach efts.sec.gov to test
# this live. Uncomment and run this cell yourself (e.g., on your own machine or
# in Colab, which should have normal internet access to sec.gov) to actually
# pull the real SEC EDGAR data:
#
# filings_8k = search_material_weakness_filings("8-K", "2020-01-01", "2026-07-17")
# filings_10k = search_material_weakness_filings("10-K", "2020-01-01", "2026-07-17")
# all_filings = filings_8k + filings_10k
# with open("../data/raw/sec_edgar_disclosures_raw.json", "w") as f:
#     json.dump(all_filings, f, indent=2)
# print(f"Saved {len(all_filings)} real SEC filings")

print("SEC EDGAR extraction not yet run. This remains an open task for RQ4.")

SEC EDGAR extraction not yet run. This remains an open task for RQ4.
